## Install API client dependencies

In [1]:
%pip install requests pandas


[notice] A new release of pip available: 22.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Configure API

In [ ]:
import os
import requests
import json
from pprint import pprint


API_BASE_URL = os.getenv(
    "FINRESEARCH_API_URL",
    "http://localhost:8000"
)

API_BASE_URL = API_BASE_URL.rstrip("/")

print("FinResearch-Qwen API")
print("=" * 60)
print("Base URL:", API_BASE_URL)

FinResearch-Qwen API
Base URL: http://localhost:8000


## Create reusable API client

In [ ]:
class FinResearchAPI:
    
    def __init__(self, base_url):
        self.base_url = base_url.rstrip("/")

    def get(self, endpoint, **kwargs):
        url = f"{self.base_url}{endpoint}"
        response = requests.get(
            url,
            timeout=30,
            **kwargs
        )
        response.raise_for_status()
        return response.json()

    def post(self, endpoint, payload=None, **kwargs):
        url = f"{self.base_url}{endpoint}"
        response = requests.post(
            url,
            json=payload,
            timeout=300,
            **kwargs
        )
        response.raise_for_status()
        return response.json()

api = FinResearchAPI(
    API_BASE_URL
)
print("API client initialized.")

API client initialized.


## Health check

In [4]:
health = api.get(
    "/health"
)

print(
    json.dumps(
        health,
        indent=2
    )
)

{
  "status": "ok",
  "service": "finresearch-qwen",
  "indexed_chunks": 833
}


## Pretty health check

In [5]:
health = api.get("/health")

print("=" * 60)
print("FINRESEARCH-QWEN SYSTEM HEALTH")
print("=" * 60)

print(
    f"Status          : {health.get('status')}"
)

print(
    f"Service         : {health.get('service')}"
)

print(
    f"Indexed chunks  : {health.get('indexed_chunks')}"
)

print("=" * 60)

FINRESEARCH-QWEN SYSTEM HEALTH
Status          : ok
Service         : finresearch-qwen
Indexed chunks  : 833


## Developer endpoint

In [6]:
developer = api.get(
    "/v1/developer"
)

print(
    json.dumps(
        developer,
        indent=2
    )
)

{
  "developer": "Roshan Jaiswal (Codecortex)",
  "contact": {
    "linkedin": "https://www.linkedin.com/in/codecortex/",
    "instagram": "https://www.instagram.com/codecortexx/",
    "github": "https://github.com/CodeCortex"
  }
}


## Check Swagger/OpenAPI

In [7]:
docs_url = (
    f"{API_BASE_URL}/docs"
)

openapi_url = (
    f"{API_BASE_URL}/openapi.json"
)

print("Swagger UI:")
print(docs_url)

print()

print("OpenAPI:")
print(openapi_url)

Swagger UI:
http://localhost:8000/docs

OpenAPI:
http://localhost:8000/openapi.json


## Check OpenAPI programmatically

In [9]:
openapi = api.get("/openapi.json")

print("API title:",openapi["info"]["title"])
print("API version:",openapi["info"]["version"])
print()

print("Available endpoints:")
for path in openapi["paths"]:
    print(" -",path)
    

API title: FinResearch-Qwen API
API version: 2.0.0

Available endpoints:
 - /health
 - /v1/developer
 - /v1/research
 - /v1/index


## Index your financial documents

In [13]:
index_request = {
    "directory": "data/raw/company_reports",
    "chunk_size": 1200,
    "chunk_overlap": 200,
    "clear": False,
}

index_response = api.post("/v1/index",index_request)
print(json.dumps(index_response, indent=2))

{
  "status": "ok",
  "documents": 1,
  "chunks": 833
}


## Verify indexing

In [14]:
health = api.get("/health")

print("=" * 60)
print("RAG KNOWLEDGE BASE")
print("=" * 60)

print("Indexed chunks:", health["indexed_chunks"])
print("=" * 60)

RAG KNOWLEDGE BASE
Indexed chunks: 1666


## First financial research request

In [ ]:
research_request = {
    "question": (
        "What was the company's revenue growth "
        "between FY2025 and FY2026? "
        "Show the calculation and explain the result."
    ),
    "use_rag": True,
    "top_k": 5,
}

research_response = api.post("/v1/research", research_request)

print(
    json.dumps(
        research_response,
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
answer = research_response.get("answer", "")
sources = research_response.get("sources", [])

print("=" * 80)
print("FINRESEARCH-QWEN ANSWER")
print("=" * 80)

print(answer)

print()
print("=" * 80)
print("SOURCES")
print("=" * 80)

for index, source in enumerate(sources, start=1):
    print(f"\n[{index}] " f"{source.get('file_name', 'Unknown')}")
    print(f"Page: {source.get('page', 'N/A')}")
    print(f"Score: {source.get('score', 'N/A')}")
    print( f"Snippet: " f"{source.get('snippet', '')[:500]}")

## Test RAG disabled

In [ ]:
no_rag_request = {
    "question": (
        "Explain what revenue growth means "
        "in financial analysis."
    ),
    "use_rag": False,
    "top_k": 5,
}


no_rag_response = api.post(
    "/v1/research",
    no_rag_request
)


print(
    no_rag_response["answer"]
)

## Compare RAG vs No-RAG

In [ ]:
question = """
What was the company's revenue growth between FY2025
and FY2026? Show the calculation using the financial
evidence available in the knowledge base.
"""

without_rag = api.post(
    "/v1/research",
    {
        "question": question,
        "use_rag": False,
        "top_k": 5,
    }
)


with_rag = api.post(
    "/v1/research",
    {
        "question": question,
        "use_rag": True,
        "top_k": 5,
    }
)


print("=" * 80)
print("WITHOUT RAG")
print("=" * 80)

print(without_rag["answer"])


print()
print("=" * 80)
print("WITH RAG")
print("=" * 80)

print(with_rag["answer"])

## Inspect retrieved sources

In [ ]:
sources = with_rag.get(
    "sources",
    []
)

print(
    f"Retrieved {len(sources)} sources"
)

print()


for index, source in enumerate(
    sources,
    start=1
):

    print("=" * 80)

    print(
        f"SOURCE {index}"
    )

    print(
        "File:",
        source.get(
            "file_name",
            "Unknown"
        )
    )

    print(
        "Page:",
        source.get(
            "page",
            "N/A"
        )
    )

    print(
        "Score:",
        source.get(
            "score",
            "N/A"
        )
    )

    print()

    print(
        source.get(
            "snippet",
            ""
        )
    )

## Create reusable research function

In [ ]:
def research(
    question,
    use_rag=True,
    top_k=5
):

    payload = {

        "question": question,

        "use_rag": use_rag,

        "top_k": top_k,
    }

    response = api.post(
        "/v1/research",
        payload
    )

    print("=" * 80)
    print("FINRESEARCH-QWEN")
    print("=" * 80)

    print()

    print(
        response.get(
            "answer",
            "No answer returned."
        )
    )

    sources = response.get(
        "sources",
        []
    )

    if sources:

        print()
        print("=" * 80)
        print(
            f"EVIDENCE ({len(sources)} sources)"
        )
        print("=" * 80)

        for i, source in enumerate(
            sources,
            start=1
        ):

            print(
                f"\n[{i}] "
                f"{source.get('file_name')}"
            )

            print(
                f"Page: "
                f"{source.get('page', 'N/A')}"
            )

            print(
                f"Score: "
                f"{source.get('score', 'N/A')}"
            )

            print(
                source.get(
                    "snippet",
                    ""
                )
            )

    return response

In [ ]:
research(
    """
    What are the major financial risks
    mentioned in the company reports?
    """
)

## Ask multiple financial questions

In [ ]:
questions = [

    """
    What was the revenue growth between
    FY2024 and FY2025?
    """,

    """
    What were the major risks mentioned
    in the company's financial reports?
    """,

    """
    What was the company's operating margin
    and how did it change?
    """,

    """
    What are the key factors affecting
    profitability?
    """,

]


for question in questions:

    research(
        question,
        use_rag=True,
        top_k=5
    )

    print("\n\n")

## Test different top_k

In [ ]:
question = """
What are the major factors affecting
the company's profitability?
"""

### top 3


In [ ]:
response_3 = api.post(
    "/v1/research",
    {
        "question": question,
        "use_rag": True,
        "top_k": 3,
    }
)

print(
    response_3["answer"]
)

### top 10

In [ ]:
response_10 = api.post(
    "/v1/research",
    {
        "question": question,
        "use_rag": True,
        "top_k": 10,
    }
)

print(
    response_10["answer"]
)

## Complete API smoke test

In [ ]:
def run_api_smoke_test():
    
    print("=" * 70)
    print("FINRESEARCH-QWEN API SMOKE TEST")
    print("=" * 70)


    health = api.get(
        "/health"
    )

    assert health["status"] == "ok"

    print("✓ Health endpoint")

    # Developer
    developer = api.get(
        "/v1/developer"
    )

    assert "developer" in developer

    print("✓ Developer endpoint")

    # OpenAPI
    openapi = api.get(
        "/openapi.json"
    )

    assert "/health" in openapi["paths"]

    assert "/v1/research" in openapi["paths"]

    assert "/v1/index" in openapi["paths"]

    print("✓ OpenAPI endpoint")

    # Research
    research_result = api.post(
        "/v1/research",
        {
            "question": (
                "Explain revenue growth."
            ),
            "use_rag": True,
            "top_k": 3,
        }
    )

    assert "answer" in research_result

    print("✓ Research endpoint")

    # Final
    print()
    print("=" * 70)
    print("ALL API TESTS PASSED")
    print("=" * 70)


run_api_smoke_test()